In [ ]:
import sys
sys.path.append("../../")
import torch
import numpy as np
import sys
!{sys.executable} -m pip install seaborn
device = torch.device("cpu")
print("imports done")

imports done


In [3]:
import sys
sys.path.append("../../")
import torch

from models.transformer import TransformerEncoderDecoder

ckpt = torch.load(
    "../../results/transformer/study1/train_best_model.pt",
    map_location="cpu"
)
print("ckpt keys:", ckpt.keys())
print("val_accuracy:", ckpt["val_accuracy"])
print("done")

ckpt keys: dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_loss', 'val_accuracy'])
val_accuracy: 7.0
done


In [4]:
import math
import torch
import torch.nn as nn
import sys
sys.path.append("../../")

from models.transformer import TransformerEncoderDecoder

# Vocab — hardcoded rules (20 tokens)
vocab = {
    '<PAD>': 0, '<SOS>': 1, '<EOS>': 2,
    '0': 3, '1': 4, '2': 5, '3': 6, '4': 7,
    '5': 8, '6': 9, '7': 10, '8': 11, '9': 12,
    '+': 13, '-': 14, '*': 15, '/': 16,
    '(': 17, ')': 18, ' ': 19
}
id2token = {v: k for k, v in vocab.items()}
print("Vocab size:", len(vocab))

# Load model
model = TransformerEncoderDecoder(
    vocab_size=20,
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=1024,
    dropout=0.0
)
ckpt = torch.load(
    "../../results/transformer/study1/train_best_model.pt",
    map_location="cpu"
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Model loaded. Val accuracy:", ckpt["val_accuracy"])

Vocab size: 20
Model loaded. Val accuracy: 7.0


In [5]:
from data.generate_controlled import generate_controlled_dataset
from data.tokenizer import create_tokenizer

# Build tokenizer
tokenizer = create_tokenizer()
print("Vocab:", tokenizer.vocab)
print("Vocab size:", len(tokenizer.vocab))

# Generate OOD samples
ood_raw = generate_controlled_dataset(
    num_samples=1000,
    num_ops_range=(4, 7),
    depth_limit=4,
    seed=42
)

print(f"\nGenerated {len(ood_raw)} samples")
print("Sample 0 keys:", ood_raw[0].keys())
print("Sample 0:", ood_raw[0])

Vocab: ['<PAD>', '<SOS>', '<EOS>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '-', '*', '/', '(', ')', ' ']
Vocab size: 20
Generating 1000 controlled samples...
Operations range: 4-7, Depth limit: 4
------------------------------------------------------------
✓ Generated 100/1000 samples
✓ Generated 200/1000 samples
✓ Generated 300/1000 samples
✓ Generated 400/1000 samples
✓ Generated 500/1000 samples
✓ Generated 600/1000 samples
✓ Generated 700/1000 samples
✓ Generated 800/1000 samples
✓ Generated 900/1000 samples
✓ Generated 1000/1000 samples

Successfully generated 1000/1000 samples

Generated 1000 samples
Sample 0 keys: dict_keys(['seed_id', 'depth', 'num_operations', 'result', 'intermediate_max', 'expression', 'input', 'output'])
Sample 0: {'seed_id': 0, 'depth': 3, 'num_operations': 4, 'result': -41, 'intermediate_max': 41, 'expression': '(((2 + 1) - 7) - (17 + 20))', 'input': '(((2 + 1) - 7) - (17 + 20))', 'output': '-41'}


In [ ]:
import torch
from data.generate_controlled import generate_controlled_dataset
from data.tokenizer import create_tokenizer

# tokenizer 
tokenizer = create_tokenizer()
vocab    = {tok: i for i, tok in enumerate(tokenizer.vocab)}
id2token = {i: tok for i, tok in enumerate(tokenizer.vocab)}
print("Vocab size:", len(vocab))

# generate ID test set (2-3 ops) 
id_raw = generate_controlled_dataset(
    num_samples=200,
    num_ops_range=(2, 3),
    depth_limit=2,
    seed=99
)

# generate OOD test set (4-7 ops) 
ood_raw = generate_controlled_dataset(
    num_samples=1000,
    num_ops_range=(4, 7),
    depth_limit=4,
    seed=42
)

# tokenize function 
def encode(text, max_len=80):
    tokens = [vocab['<SOS>']]
    for ch in text:
        if ch in vocab:
            tokens.append(vocab[ch])
    tokens.append(vocab['<EOS>'])
    # pad
    tokens += [vocab['<PAD>']] * (max_len - len(tokens))
    return torch.tensor(tokens[:max_len], dtype=torch.long)

def prepare_samples(raw):
    samples = []
    for s in raw:
        src = encode(s['input'])
        tgt = encode(s['output'])
        samples.append((
            src,
            tgt,
            s['input'],
            s['output'],
            s['num_operations']
        ))
    return samples

id_data  = prepare_samples(id_raw)
ood_data = prepare_samples(ood_raw)

print(f"ID  samples: {len(id_data)}")
print(f"OOD samples: {len(ood_data)}")

# quick sanity check 
src, tgt, expr, ans, nops = ood_data[0]
print(f"\nSample 0: {expr} → {ans} ({nops} ops)")
print(f"src shape: {src.shape}")
print(f"tgt shape: {tgt.shape}")
print(f"decoded src: {''.join([id2token[t.item()] for t in src if t.item() not in [0,1,2]])}")
print(f"decoded tgt: {''.join([id2token[t.item()] for t in tgt if t.item() not in [0,1,2]])}")

Vocab size: 20
Generating 200 controlled samples...
Operations range: 2-3, Depth limit: 2
------------------------------------------------------------
✓ Generated 100/200 samples
✓ Generated 200/200 samples

Successfully generated 200/200 samples
Generating 1000 controlled samples...
Operations range: 4-7, Depth limit: 4
------------------------------------------------------------
✓ Generated 100/1000 samples
✓ Generated 200/1000 samples
✓ Generated 300/1000 samples
✓ Generated 400/1000 samples
✓ Generated 500/1000 samples
✓ Generated 600/1000 samples
✓ Generated 700/1000 samples
✓ Generated 800/1000 samples
✓ Generated 900/1000 samples
✓ Generated 1000/1000 samples

Successfully generated 1000/1000 samples
ID  samples: 200
OOD samples: 1000

Sample 0: (((2 + 1) - 7) - (17 + 20)) → -41 (4 ops)
src shape: torch.Size([80])
tgt shape: torch.Size([80])
decoded src: (((2 + 1) - 7) - (17 + 20))
decoded tgt: -41


In [8]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import json
import os
from collections import defaultdict

import sys
import io

# Force full output
class Unbuffered:
    def __init__(self, stream):
        self.stream = stream
    def write(self, data):
        self.stream.write(data)
        self.stream.flush()
    def writelines(self, lines):
        self.stream.writelines(lines)
        self.stream.flush()
    def __getattr__(self, attr):
        return getattr(self.stream, attr)

sys.stdout = Unbuffered(sys.stdout)

RESULTS_DIR = "../../experiments/results"
os.makedirs(f"{RESULTS_DIR}/attention_maps",      exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/cross_attention",     exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/head_ablation",       exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/operation_breakdown", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/failure_traces",      exist_ok=True)


# HELPERS
def decode_tokens(token_ids):
    return [id2token.get(t.item(), "?") for t in token_ids
            if t.item() not in [vocab['<PAD>']]]

def detect_op_type(expr_str):
    counts = {"+": 0, "-": 0, "*": 0, "/": 0}
    for ch in expr_str:
        if ch in counts:
            counts[ch] += 1
    return max(counts, key=counts.get) if any(counts.values()) else "mixed"

def run_forward(model, src, tgt):
    """Standard forward — no attention."""
    with torch.no_grad():
        tgt_in = tgt[:-1].unsqueeze(0)
        src_in = src.unsqueeze(0)
        logits = model(src_in, tgt_in)
    return logits

def is_correct(logits, tgt):
    preds  = logits.argmax(dim=-1)[0]
    tgt_gt = tgt[1:len(preds)+1]
    # trim to EOS
    eos = vocab['<EOS>']
    pred_list = []
    gt_list   = []
    for p, g in zip(preds, tgt_gt):
        pred_list.append(p.item())
        gt_list.append(g.item())
        if g.item() == eos:
            break
    return pred_list == gt_list

# CP1 — ENCODER SELF-ATTENTION HEATMAPS
def run_cp1(model, id_data, ood_data, n=3):
    print("\n" + "="*60)
    print("CP1: ENCODER SELF-ATTENTION HEATMAPS")
    print("="*60)

    for split_name, samples in [("ID", id_data[:n]), ("OOD", ood_data[:n])]:
        for i, (src, tgt, expr_str, ans_str, nops) in enumerate(samples):
            src_in = src.unsqueeze(0)
            tgt_in = tgt[:-1].unsqueeze(0)

            with torch.no_grad():
                logits, attn_dict = model(
                    src_in, tgt_in, return_attention=True
                )

            enc_weights = attn_dict['encoder_attention']
            # enc_weights: list of 3 tensors [1, heads, src, src]

            src_labels = decode_tokens(src)
            n_layers   = len(enc_weights)
            n_heads    = enc_weights[0].shape[1]

            fig, axes = plt.subplots(
                n_layers, n_heads,
                figsize=(n_heads * 2, n_layers * 2)
            )
            fig.suptitle(
                f"CP1 Encoder Self-Attention [{split_name} ex{i+1}]\n"
                f"{expr_str[:50]}",
                fontsize=9
            )

            for l in range(n_layers):
                for h in range(n_heads):
                    ax  = axes[l][h]
                    # trim to actual non-pad length
                    seq_len = len(src_labels)
                    attn    = enc_weights[l][0, h, :seq_len, :seq_len].numpy()

                    sns.heatmap(
                        attn, ax=ax,
                        xticklabels=src_labels,
                        yticklabels=src_labels,
                        cmap="Blues",
                        vmin=0, vmax=1,
                        cbar=False,
                        square=True
                    )
                    ax.set_title(f"L{l+1}H{h+1}", fontsize=6)
                    ax.tick_params(labelsize=4)

            plt.tight_layout()
            path = f"{RESULTS_DIR}/attention_maps/cp1_{split_name}_ex{i+1}.png"
            plt.savefig(path, dpi=120, bbox_inches="tight")
            plt.close()
            print(f"  Saved: {path}")

    print("CP1 complete ✓")


# CP2 — CROSS-ATTENTION PER DECODING STEP 
def run_cp2(model, id_data, ood_data, n=3):
    print("\n" + "="*60)
    print("CP2: CROSS-ATTENTION PER DECODING STEP")
    print("="*60)

    for split_name, samples in [("ID", id_data[:n]), ("OOD", ood_data[:n])]:
        for i, (src, tgt, expr_str, ans_str, nops) in enumerate(samples):
            src_in = src.unsqueeze(0)
            tgt_in = tgt[:-1].unsqueeze(0)

            with torch.no_grad():
                logits, attn_dict = model(
                    src_in, tgt_in, return_attention=True
                )

            dec_weights = attn_dict['decoder_attention']
            # dec_weights: list of 3 dicts with 'cross_attention'
            # each: [1, heads, tgt_len, src_len]

            src_labels = decode_tokens(src)
            tgt_labels = decode_tokens(tgt[:-1])

            # average across layers and heads
            cross_stack = torch.stack(
                [d['cross_attention'][0] for d in dec_weights], dim=0
            )  # [layers, heads, tgt, src]
            avg_cross = cross_stack.mean(dim=(0, 1)).numpy()

            # trim to actual lengths
            t_len = len(tgt_labels)
            s_len = len(src_labels)
            avg_cross = avg_cross[:t_len, :s_len]

            fig, ax = plt.subplots(
                figsize=(s_len * 0.4 + 2, t_len * 0.35 + 2)
            )
            sns.heatmap(
                avg_cross, ax=ax,
                xticklabels=src_labels,
                yticklabels=tgt_labels,
                cmap="Oranges",
                vmin=0, vmax=avg_cross.max(),
                cbar=True
            )
            ax.set_xlabel("Source tokens", fontsize=8)
            ax.set_ylabel("Decoding steps", fontsize=8)
            ax.set_title(
                f"CP2 Cross-Attention [{split_name} ex{i+1}]\n"
                f"{expr_str[:40]} → {ans_str}",
                fontsize=8
            )
            plt.tight_layout()
            path = f"{RESULTS_DIR}/cross_attention/cp2_{split_name}_ex{i+1}.png"
            plt.savefig(path, dpi=120, bbox_inches="tight")
            plt.close()
            print(f"  Saved: {path}")

    print("CP2 complete ✓")


# CP3 — HEAD ABLATION BAR CHART
def ablate_and_eval(model, data, layer_idx, head_idx, component):
    """Zero out one head via hook, evaluate accuracy."""
    correct = 0

    def hook_fn(module, input, output):
        attn_out = output[0].clone()
        d_model  = attn_out.shape[-1]
        head_dim = d_model // module.num_heads
        attn_out[:, :, head_idx*head_dim:(head_idx+1)*head_dim] = 0
        return (attn_out,) + output[1:]

    if component == "encoder":
        handle = model.transformer.encoder.layers[layer_idx]\
                     .self_attn.register_forward_hook(hook_fn)
    else:
        handle = model.transformer.decoder.layers[layer_idx]\
                     .multihead_attn.register_forward_hook(hook_fn)

    with torch.no_grad():
        for src, tgt, *_ in data:
            logits = run_forward(model, src, tgt)
            if is_correct(logits, tgt):
                correct += 1

    handle.remove()
    return correct / len(data)


def run_cp3(model, id_data, ood_data):
    print("\n" + "="*60)
    print("CP3: HEAD ABLATION")
    print("="*60)

    print("  Computing baselines...")
    baseline_id  = sum(
        is_correct(run_forward(model, s, t), t)
        for s, t, *_ in id_data
    ) / len(id_data)
    baseline_ood = sum(
        is_correct(run_forward(model, s, t), t)
        for s, t, *_ in ood_data
    ) / len(ood_data)
    print(f"  Baseline ID:  {baseline_id:.4f}")
    print(f"  Baseline OOD: {baseline_ood:.4f}")

    results = []
    for comp in ["encoder", "decoder_cross"]:
        for l in range(3):
            for h in range(8):
                label   = f"{comp[0].upper()}L{l+1}H{h+1}"
                acc_id  = ablate_and_eval(model, id_data,  l, h, comp)
                acc_ood = ablate_and_eval(model, ood_data, l, h, comp)
                id_drop  = baseline_id  - acc_id
                ood_drop = baseline_ood - acc_ood
                delta    = ood_drop - id_drop
                results.append({
                    "label": label, "id_drop": id_drop,
                    "ood_drop": ood_drop, "delta": delta
                })
                print(f"  {label}: ID↓{id_drop:.3f} OOD↓{ood_drop:.3f} Δ{delta:.3f}")

    with open(f"{RESULTS_DIR}/head_ablation/cp3_results.json", "w") as f:
        json.dump(results, f, indent=2)

    results_sorted = sorted(results, key=lambda x: x["delta"], reverse=True)
    labels   = [r["label"]    for r in results_sorted]
    id_drops = [r["id_drop"]  for r in results_sorted]
    ood_drops= [r["ood_drop"] for r in results_sorted]
    deltas   = [r["delta"]    for r in results_sorted]
    x        = np.arange(len(labels))
    w        = 0.35

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 10))

    ax1.bar(x-w/2, id_drops,  w, label="ID drop",  color="#4C72B0", alpha=0.85)
    ax1.bar(x+w/2, ood_drops, w, label="OOD drop", color="#DD8452", alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=90, fontsize=7)
    ax1.set_ylabel("Accuracy Drop")
    ax1.set_title("CP3 — Head Ablation: ID Drop vs OOD Drop")
    ax1.legend()
    ax1.axhline(0, color="black", linewidth=0.5)

    colors = ["#c0392b" if d > 0.01 else "#2ecc71" for d in deltas]
    ax2.bar(x, deltas, color=colors, alpha=0.85)
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels, rotation=90, fontsize=7)
    ax2.set_ylabel("Δ (OOD drop − ID drop)")
    ax2.set_title("CP3 — Compositional Head Importance (red = Δ > 0.01)")
    ax2.axhline(0,    color="black", linewidth=0.8)
    ax2.axhline(0.01, color="red",   linewidth=0.8,
                linestyle="--", label="Δ=0.01 threshold")
    red_p   = mpatches.Patch(color="#c0392b", label="Compositional (Δ>0.01)")
    green_p = mpatches.Patch(color="#2ecc71", label="General (Δ≤0.01)")
    ax2.legend(handles=[red_p, green_p])

    plt.tight_layout()
    path = f"{RESULTS_DIR}/head_ablation/cp3_ablation_chart.png"
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")
    print("CP3 complete ✓")
    return results


# CP4 — FAILURE BREAKDOWN BY OPERATION TYPE
def run_cp4(model, ood_data):
    print("\n" + "="*60)
    print("CP4: FAILURE BREAKDOWN BY OPERATION TYPE")
    print("="*60)

    op_correct = defaultdict(int)
    op_total   = defaultdict(int)

    for src, tgt, expr_str, ans_str, nops in ood_data:
        op = detect_op_type(expr_str)
        logits = run_forward(model, src, tgt)
        op_correct[op] += int(is_correct(logits, tgt))
        op_total[op]   += 1

    table = {}
    print(f"\n  {'Op':<6} {'Correct':<10} {'Total':<10} {'Accuracy':<10}")
    print(f"  {'-'*36}")
    for op in ["+", "-", "*", "/"]:
        tot = op_total.get(op, 0)
        cor = op_correct.get(op, 0)
        acc = cor / tot if tot > 0 else 0.0
        table[op] = {"correct": cor, "total": tot, "accuracy": round(acc, 4)}
        print(f"  {op:<6} {cor:<10} {tot:<10} {acc:.4f}")

    with open(f"{RESULTS_DIR}/operation_breakdown/cp4_results.json","w") as f:
        json.dump(table, f, indent=2)

    ops    = ["+", "-", "*", "/"]
    accs   = [table[op]["accuracy"] for op in ops]
    colors = ["#2ecc71" if a > 0.05 else "#c0392b" for a in accs]

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(ops, accs, color=colors, alpha=0.85, width=0.5)
    ax.set_ylim(0, max(accs) * 1.4 + 0.02)
    ax.set_xlabel("Dominant Operation Type")
    ax.set_ylabel("OOD Accuracy")
    ax.set_title("CP4 — OOD Accuracy by Operation Type")
    ax.axhline(0.05, color="gray", linestyle="--",
               linewidth=0.8, label="5% baseline")
    for bar, acc in zip(bars, accs):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.002,
            f"{acc:.3f}", ha="center", va="bottom", fontsize=11
        )
    ax.legend()
    plt.tight_layout()
    path = f"{RESULTS_DIR}/operation_breakdown/cp4_op_breakdown.png"
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"\n  Saved: {path}")
    print("CP4 complete ✓")
    return table


# CP5 — FAILURE ONSET TRACE
def run_cp5(model, ood_data, n_failures=5):
    print("\n" + "="*60)
    print("CP5: FAILURE ONSET TRACES")
    print("="*60)

    traces         = []
    failures_found = 0

    for src, tgt, expr_str, ans_str, nops in ood_data:
        if failures_found >= n_failures:
            break

        logits = run_forward(model, src, tgt)
        if is_correct(logits, tgt):
            continue

        preds  = logits.argmax(dim=-1)[0]
        tgt_gt = tgt[1:]
        eos    = vocab['<EOS>']

        gt_toks   = []
        pred_toks = []
        for g, p in zip(tgt_gt, preds):
            gt_toks.append(id2token.get(g.item(), "?"))
            pred_toks.append(id2token.get(p.item(), "?"))
            if g.item() == eos:
                break

        trace = {
            "expression":       expr_str,
            "expected":         ans_str,
            "num_ops":          nops,
            "steps":            [],
            "first_error_step": None
        }

        print(f"\n  Example {failures_found+1}: {expr_str} → {ans_str}")
        for step, (gt, pred) in enumerate(zip(gt_toks, pred_toks)):
            correct  = gt == pred
            is_first = not correct and trace["first_error_step"] is None
            if is_first:
                trace["first_error_step"] = step
            trace["steps"].append({
                "step": step, "expected": gt,
                "predicted": pred, "correct": correct
            })
            marker = "✓" if correct else "✗" + (" [FIRST ERROR]" if is_first else "")
            print(f"    Step {step}: expected '{gt}' | predicted '{pred}' {marker}")

        traces.append(trace)
        failures_found += 1

    with open(f"{RESULTS_DIR}/failure_traces/cp5_traces.json", "w") as f:
        json.dump(traces, f, indent=2)

    # FIXED: include step-0 failures (height=0 bars were invisible before) 
    if traces:
        first_errors = [t["first_error_step"] if t["first_error_step"] is not None else 0
                        for t in traces]
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(range(len(first_errors)), first_errors,
               color="#9b59b6", alpha=0.85)
        ax.set_xlabel("Failure Example")
        ax.set_ylabel("First Error Step (0 = immediate failure)")
        ax.set_ylim(-0.1, max(first_errors) + 1.5)
        ax.set_xticks(range(len(first_errors)))
        ax.set_title("CP5 — Which Decoding Step First Fails on OOD")
        for i, v in enumerate(first_errors):
            ax.text(i, v + 0.05, f"Step {v}", ha='center', fontsize=9)
        plt.tight_layout()
        path = f"{RESULTS_DIR}/failure_traces/cp5_failure_onset.png"
        plt.savefig(path, dpi=120, bbox_inches="tight")
        plt.close()
        print(f"\n  Saved: {path}")

    print("CP5 complete ✓")
    return traces


# RUN ALL
print("Starting full attention analysis...")
run_cp1(model, id_data, ood_data, n=3)
run_cp2(model, id_data, ood_data, n=3)
cp3_results = run_cp3(model, id_data, ood_data)
run_cp4(model, ood_data)
run_cp5(model, ood_data, n_failures=5)
print("\n ALL 5 CHECKPOINTS COMPLETE")
print(f"Results saved to: {RESULTS_DIR}")

Matplotlib is building the font cache; this may take a moment.


Starting full attention analysis...

CP1: ENCODER SELF-ATTENTION HEATMAPS
  Saved: ../../experiments/results/attention_maps/cp1_ID_ex1.png
  Saved: ../../experiments/results/attention_maps/cp1_ID_ex2.png
  Saved: ../../experiments/results/attention_maps/cp1_ID_ex3.png
  Saved: ../../experiments/results/attention_maps/cp1_OOD_ex1.png
  Saved: ../../experiments/results/attention_maps/cp1_OOD_ex2.png
  Saved: ../../experiments/results/attention_maps/cp1_OOD_ex3.png
CP1 complete ✓

CP2: CROSS-ATTENTION PER DECODING STEP
  Saved: ../../experiments/results/cross_attention/cp2_ID_ex1.png
  Saved: ../../experiments/results/cross_attention/cp2_ID_ex2.png
  Saved: ../../experiments/results/cross_attention/cp2_ID_ex3.png
  Saved: ../../experiments/results/cross_attention/cp2_OOD_ex1.png
  Saved: ../../experiments/results/cross_attention/cp2_OOD_ex2.png
  Saved: ../../experiments/results/cross_attention/cp2_OOD_ex3.png
CP2 complete ✓

CP3: HEAD ABLATION
  Computing baselines...
  Baseline ID:  0.0

In [ ]:
# Cell: Analyze all 5 checkpoints programmatically
import json
import os
from pathlib import Path

RESULTS_DIR = "../../experiments/results"

print("="*70)
print("COMPREHENSIVE CHECKPOINT ANALYSIS")
print("="*70)

# CP3 — HEAD ABLATION RESULTS
print("\n" + "="*70)
print("CP3 — HEAD ABLATION (Compositional Heads)")
print("="*70)
try:
    with open(f"{RESULTS_DIR}/head_ablation/cp3_results.json") as f:
        cp3_data = json.load(f)
    
    # Sort by delta (OOD drop - ID drop)
    sorted_heads = sorted(cp3_data, key=lambda x: x['delta'], reverse=True)
    
    print(f"\n TOP 10 COMPOSITIONAL HEADS (highest delta):")
    for i, head in enumerate(sorted_heads[:10], 1):
        print(f"  {i}. {head['label']}: Δ={head['delta']:.4f} | ID↓{head['id_drop']:.4f} OOD↓{head['ood_drop']:.4f}")
    
    print(f"\n TOP 10 GENERAL HEADS (lowest delta):")
    for i, head in enumerate(sorted_heads[-10:], 1):
        print(f"  {i}. {head['label']}: Δ={head['delta']:.4f} | ID↓{head['id_drop']:.4f} OOD↓{head['ood_drop']:.4f}")
except Exception as e:
    print(f"  Error loading CP3: {e}")

# CP4 — OPERATION BREAKDOWN
print("\n" + "="*70)
print("CP4 — OPERATION BREAKDOWN (Accuracy by Op Type)")
print("="*70)
try:
    with open(f"{RESULTS_DIR}/operation_breakdown/cp4_results.json") as f:
        cp4_data = json.load(f)
    
    print(f"\n{'Op':<6} {'Correct':<12} {'Total':<12} {'Accuracy':<12}")
    print(f"{'-'*42}")
    for op in ["+", "-", "*", "/"]:
        if op in cp4_data:
            cor = cp4_data[op]['correct']
            tot = cp4_data[op]['total']
            acc = cp4_data[op]['accuracy']
            print(f"{op:<6} {cor:<12} {tot:<12} {acc:.4f}")
    
    worst_op = min(cp4_data.items(), key=lambda x: x[1]['accuracy'])
    print(f"\n  WORST performing operation: '{worst_op[0]}' ({worst_op[1]['accuracy']:.2%})")
except Exception as e:
    print(f"  Error loading CP4: {e}")

# CP5 — FAILURE ONSET TRACES
print("\n" + "="*70)
print("CP5 — FAILURE ONSET TRACES (When do errors first occur?)")
print("="*70)
try:
    with open(f"{RESULTS_DIR}/failure_traces/cp5_traces.json") as f:
        cp5_data = json.load(f)
    
    print(f"\n Analyzing {len(cp5_data)} failure examples:\n")
    for i, trace in enumerate(cp5_data, 1):
        first_error = trace['first_error_step']
        print(f"  Example {i}:")
        print(f"    Input:  {trace['expression']}")
        print(f"    Target: {trace['expected']}")
        print(f"    Ops:    {trace['num_ops']}")
        print(f"     First error at step: {first_error}")
        
        # Show the error step detail
        if first_error is not None and first_error < len(trace['steps']):
            err_step = trace['steps'][first_error]
            print(f"       Expected '{err_step['expected']}' but got '{err_step['predicted']}'")
        print()
    
    # Statistics
    first_errors = [t['first_error_step'] for t in cp5_data if t['first_error_step'] is not None]
    if first_errors:
        print(f" FAILURE ONSET STATISTICS:")
        print(f"  Mean step: {sum(first_errors)/len(first_errors):.1f}")
        print(f"  Min step:  {min(first_errors)}")
        print(f"  Max step:  {max(first_errors)}")
except Exception as e:
    print(f"  Error loading CP5: {e}")

# List generated images
print("\n" + "="*70)
print(" GENERATED VISUALIZATIONS")
print("="*70)
for folder in ['attention_maps', 'cross_attention', 'head_ablation', 'operation_breakdown', 'failure_traces']:
    path = Path(f"{RESULTS_DIR}/{folder}")
    if path.exists():
        images = list(path.glob("*.png"))
        print(f"\n{folder}/ ({len(images)} files)")
        for img in sorted(images):
            print(f"  ✓ {img.name}")

COMPREHENSIVE CHECKPOINT ANALYSIS

CP3 — HEAD ABLATION (Compositional Heads)

 TOP 10 COMPOSITIONAL HEADS (highest delta):
  1. EL3H2: Δ=0.0160 | ID↓-0.0150 OOD↓0.0010
  2. DL1H6: Δ=0.0150 | ID↓-0.0150 OOD↓0.0000
  3. DL2H4: Δ=0.0150 | ID↓-0.0150 OOD↓0.0000
  4. DL3H7: Δ=0.0150 | ID↓-0.0150 OOD↓0.0000
  5. EL2H3: Δ=0.0110 | ID↓-0.0100 OOD↓0.0010
  6. EL2H6: Δ=0.0110 | ID↓-0.0100 OOD↓0.0010
  7. EL2H1: Δ=0.0100 | ID↓-0.0100 OOD↓0.0000
  8. EL3H8: Δ=0.0100 | ID↓-0.0100 OOD↓0.0000
  9. DL1H7: Δ=0.0100 | ID↓-0.0100 OOD↓0.0000
  10. DL1H8: Δ=0.0100 | ID↓-0.0100 OOD↓0.0000

 TOP 10 GENERAL HEADS (lowest delta):
  1. EL1H4: Δ=0.0000 | ID↓0.0000 OOD↓0.0000
  2. EL1H7: Δ=0.0000 | ID↓0.0000 OOD↓0.0000
  3. EL1H8: Δ=0.0000 | ID↓0.0000 OOD↓0.0000
  4. EL3H6: Δ=0.0000 | ID↓0.0000 OOD↓0.0000
  5. DL2H2: Δ=0.0000 | ID↓0.0000 OOD↓0.0000
  6. DL2H3: Δ=0.0000 | ID↓0.0000 OOD↓0.0000
  7. DL2H8: Δ=0.0000 | ID↓0.0000 OOD↓0.0000
  8. DL3H2: Δ=0.0000 | ID↓0.0000 OOD↓0.0000
  9. DL3H3: Δ=0.0000 | ID↓0.0000 OO